In [16]:
#Potrzebne na początek pracy ze Sparkiem:

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("okienka").getOrCreate()
spark

In [2]:
czujnik_temperatury = ((12.5, "2019-01-02 12:00:00"),
(17.6, "2019-01-02 12:00:20"),
(14.6,  "2019-01-02 12:00:30"),
(22.9,  "2019-01-02 12:01:15"),
(17.4,  "2019-01-02 12:01:30"),
(25.8,  "2019-01-02 12:03:25"),
(27.1,  "2019-01-02 12:02:40"),
)

In [4]:
#Ustalenie struktury tabeli, na której chcemy działać:

from pyspark.sql.functions import to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

#Obiekt typu Struct, do którego wkładam listę, która powinna zawierać nasze pola:
schema = StructType([
    StructField("temperatura", DoubleType(), True),
    StructField("czas", StringType(), True),
])
#^^^Nazwa zmiennej i typ jako obiekt ze Sparka + (True) czy dane pole może być wartości pusta czy nie

##Schemat DDL (alternatywna opcja)
SCHEMA = """
temperatura DOUBLE, czas STRING
"""

In [5]:
df = (spark.createDataFrame(czujnik_temperatury, schema=schema)
      .withColumn("czas", to_timestamp("czas")))

df.printSchema()

root
 |-- temperatura: double (nullable = true)
 |-- czas: timestamp (nullable = true)



In [6]:
df = (spark.createDataFrame(czujnik_temperatury, schema=SCHEMA)
      .withColumn("czas", to_timestamp("czas")))

df.printSchema()

# Działa identycznie przy SCHEMA

root
 |-- temperatura: double (nullable = true)
 |-- czas: timestamp (nullable = true)



In [7]:
df.show(3)

+-----------+-------------------+
|temperatura|               czas|
+-----------+-------------------+
|       12.5|2019-01-02 12:00:00|
|       17.6|2019-01-02 12:00:20|
|       14.6|2019-01-02 12:00:30|
+-----------+-------------------+
only showing top 3 rows



In [9]:
#Możemy na Sparku pracować jak na SQL:

df.createOrReplaceTempView("df")

#Zrzucamy do widoku tymczasowego (tabela nie na stałe, możemy zdecydować, którzy użytkownicy go widza, czyli nie
#bazuja na surowych danych, tylko na przykladowych agregacjach, trzymamy kod generujacy widok

In [10]:
spark.sql("select czas, temperatura from df where temperatura > 21").show(5)

#Widok zostaje zapisany do Sparkowego data frame

+-------------------+-----------+
|               czas|temperatura|
+-------------------+-----------+
|2019-01-02 12:01:15|       22.9|
|2019-01-02 12:03:25|       25.8|
|2019-01-02 12:02:40|       27.1|
+-------------------+-----------+



In [11]:
# Thumbling window - okno przesuwne

import pyspark.sql.functions as F

df2 = df.groupBy(F.window("czas","30 seconds")).count()
df2.show(truncate=False)

+------------------------------------------+-----+
|window                                    |count|
+------------------------------------------+-----+
|{2019-01-02 12:00:00, 2019-01-02 12:00:30}|2    |
|{2019-01-02 12:00:30, 2019-01-02 12:01:00}|1    |
|{2019-01-02 12:01:00, 2019-01-02 12:01:30}|1    |
|{2019-01-02 12:01:30, 2019-01-02 12:02:00}|1    |
|{2019-01-02 12:03:00, 2019-01-02 12:03:30}|1    |
|{2019-01-02 12:02:30, 2019-01-02 12:03:00}|1    |
+------------------------------------------+-----+



In [12]:
import pyspark.sql.functions as F

df2 = df.groupBy(F.window("czas","20 seconds")).count()
df2.show(truncate=False)

+------------------------------------------+-----+
|window                                    |count|
+------------------------------------------+-----+
|{2019-01-02 12:00:00, 2019-01-02 12:00:20}|1    |
|{2019-01-02 12:00:20, 2019-01-02 12:00:40}|2    |
|{2019-01-02 12:01:00, 2019-01-02 12:01:20}|1    |
|{2019-01-02 12:01:20, 2019-01-02 12:01:40}|1    |
|{2019-01-02 12:03:20, 2019-01-02 12:03:40}|1    |
|{2019-01-02 12:02:40, 2019-01-02 12:03:00}|1    |
+------------------------------------------+-----+



In [13]:
import pyspark.sql.functions as F

df2 = df.groupBy(F.window("czas","50 seconds")).count()
df2.show(truncate=False)

# Różne przedziały, tutaj 50 sekund

+------------------------------------------+-----+
|window                                    |count|
+------------------------------------------+-----+
|{2019-01-02 12:00:00, 2019-01-02 12:00:50}|3    |
|{2019-01-02 12:00:50, 2019-01-02 12:01:40}|2    |
|{2019-01-02 12:03:20, 2019-01-02 12:04:10}|1    |
|{2019-01-02 12:02:30, 2019-01-02 12:03:20}|1    |
+------------------------------------------+-----+



In [14]:
spark.stop()

In [ ]:
# Żeby zrobić strumień w Spark, potrzeba:
#1. określić ŹRÓDŁO strumienia
#2. określić transformację ramki która uzyskamy jako strumień (może być trywialna = nie rób nic)
#3. każdy strumień musi mieć miejsce, w którym ten strumień będzie uchodził, ZAKOŃCZENIE (baza danych, plik, okno terminala -
#komunikat tekstowy).

In [17]:
# Rate — źródło testowe

df = spark.readStream.format("rate").option("rowsPerSecond", 1).load()

In [ ]:
%%file streamrate.py
## uruchom przez spark-submit streamrate.py

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)


query = (df.writeStream 
    .format("console") 
    .outputMode("append") 
    .option("truncate", False) 
    .start()
) 

query.awaitTermination()

In [18]:
# Jak zrobić, żeby kod wyświetlał się w notatniku?

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()


df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

query = (df.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

Batch ID: 0
+---------+-----+
|timestamp|value|
+---------+-----+
+---------+-----+

Batch ID: 1
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-26 16:37:48.605|0    |
+-----------------------+-----+

Batch ID: 2
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-26 16:37:49.605|1    |
+-----------------------+-----+

Batch ID: 3
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-26 16:37:50.605|2    |
+-----------------------+-----+

Batch ID: 4
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-26 16:37:51.605|3    |
+-----------------------+-----+

Batch ID: 5
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-26 16:37:52.605|4    |
+-----------------------+-----+

Batch ID: 0
+----+-----------+
|czas|temperatura|


In [19]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()

from pyspark.sql.functions import col, expr

df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )

query = (stream.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

#Wynik generuje się na górze tego kodu ^^^

In [20]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()

from pyspark.sql.functions import col, expr

df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )

stream_filtered = stream.filter(col("temperatura") > 26)

query = (stream_filtered.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

# Wynik generuje się tak samo na górze, wyświetla tylko te, które spełniaja założenie

In [21]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, when

spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=10):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()



df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )

stream_anomaly = stream.withColumn(
    "anomaly",
    when(col("temperatura") > 24.5, "TAK").otherwise("NIE")
)

query = (stream_anomaly.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

In [22]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import window

spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=20):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()


df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )

# Konwertuj 'temperatura' na typ Double, aby agregacja była możliwa
stream = stream.withColumn("temperatura", col("temperatura").cast("double"))

tumbling_window = (stream
    .groupBy(window(col("czas"), "10 seconds"))
    .avg("temperatura")
)

query = (tumbling_window.writeStream 
    .format("console") 
    .outputMode("complete")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

# nie można użyć outputMode "append"

In [23]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import window

spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=30):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()


df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )
# Konwertuj 'temperatura' na typ Double, aby agregacja była możliwa
stream = stream.withColumn("temperatura", col("temperatura").cast("double"))


sliding = (stream.groupBy(window(col("czas"), "10 seconds", "5 seconds")) #mamy dwa parametry: jak długie okno i co ile uruchamiać
           .avg("temperatura")
          )

query = (sliding.writeStream 
    .format("console") 
    .outputMode("complete")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

In [ ]:
%%file generator.py
# generator.py
import json, os, random, time
from datetime import datetime, timedelta

output_dir = "data/stream"
os.makedirs(output_dir, exist_ok=True)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

# Simulate file-based streaming
while True:
    batch = [generate_transaction() for _ in range(2)]
    filename = f"{output_dir}/events_{int(time.time())}.json"
    with open(filename, "w") as f:
        for e in batch:
            f.write(json.dumps(e) + "\n")
    print(f"Wrote: {filename}")
    time.sleep(5)

# nie zapisujemy pliku JSON i zapisywać do Kafki, tylko generujemy x transakcji za jednym razem i zapisujemy to pliku

In [24]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import from_json, to_timestamp
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("jsonDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

tx_schema = StructType([
    StructField("tx_id",     StringType()),
    StructField("user_id",   StringType()),
    StructField("amount",    DoubleType()),
    StructField("store",     StringType()),
    StructField("category",  StringType()),
    StructField("timestamp", StringType()),
])

batch_counter = {"count": 0}

def process_batch(df, batch_id, tstop=5):
    batch_counter["count"] += 1
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()
    


stream = (spark.readStream
          .schema(tx_schema)
          .json("data/stream"))

query = (stream.writeStream
         .format("console")
         .foreachBatch(process_batch)
         .start())

In [ ]:
# Na tym procesie wygenerować ramki z Lab 2 Zdanie 2.1 - Liczba transakcji i suma przychodów per sklep
# Zamiast ogladac wszystkie 10 transakcji, ktore przyszly, chcemy widziec agregat ile w danym sklepie bylo przypadkow,
# sumy amount, ilu uzytkownikow w sklepie
# Co się dzieje w sklepach co 5 sekund

In [ ]:
# Różnica miedzy mam cala tabelke i chce ja przetworzyc vs co sie dzieje w sklepach, jak mam te dane na biezaco

In [ ]:
#from pyspark.sql.functions import count, sum as _sum, avg, round as _round

#store_summary = (
#    df.groupBy("store")
#    .agg(
#        count("tx_id").alias("liczba_tx"),
#        _round(_sum("amount"), 2).alias("suma_PLN"),
#        _round(avg("amount"), 2).alias("srednia_PLN"),
#    )
#    .orderBy("store")
#)
#store_summary.show()

In [25]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import col, to_timestamp, window, count, sum as _sum, approx_count_distinct, round as _round
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("StreamingLabSummary").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

tx_schema = StructType([
    StructField("tx_id",     StringType()),
    StructField("user_id",   StringType()),
    StructField("amount",    DoubleType()),
    StructField("store",     StringType()),
    StructField("category",  StringType()),
    StructField("timestamp", StringType()),
])

batch_counter = {"count": 0}

def process_batch(df, batch_id, tstop=5):
    batch_counter["count"] += 1
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()

stream = (spark.readStream
          .schema(tx_schema)
          .json("data/stream"))

tumbling_window = (stream
    .withColumn("event_time", to_timestamp(col("timestamp"))) 
    .groupBy(
        window(col("event_time"), "5 seconds"), 
        col("store")
    )
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_przychodow"),
        approx_count_distinct("user_id").alias("unikalni_uzytkownicy")
    )
)

query = (tumbling_window.writeStream 
    .format("console") 
    .outputMode("complete")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

In [ ]:
# Potem kolejne: agregat czasowy, jak transakcje generuja sie w czasie - skoro mamy transakcje co 5 sekund, to
# jakie mamy własności po 10 sekundach - ile średnio zarabiamy, klienci, suma transakcji po czasie

In [26]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import col, to_timestamp, window, count, sum as _sum, avg, approx_count_distinct, round as _round
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Streaming10sSummary").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

tx_schema = StructType([
    StructField("tx_id",     StringType()),
    StructField("user_id",   StringType()),
    StructField("amount",    DoubleType()),
    StructField("store",     StringType()),
    StructField("category",  StringType()),
    StructField("timestamp", StringType()),
])

batch_counter = {"count": 0}

def process_batch(df, batch_id, tstop=5):
    batch_counter["count"] += 1
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()

stream = (spark.readStream
          .schema(tx_schema)
          .json("data/stream"))

summary_10s = (stream
    .withColumn("event_time", to_timestamp(col("timestamp"))) 
    .groupBy(
        window(col("event_time"), "10 seconds"),
        col("store")
    )
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_przychodow"),
        _round(avg("amount"), 2).alias("srednia_wartosc_tx"),
        approx_count_distinct("user_id").alias("unikalni_uzytkownicy")
    )
)

query = (summary_10s.writeStream 
    .format("console") 
    .outputMode("complete")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)